# Actor-Critic: Learning a Baseline Online

**Family:** Reinforcement learning | **Level:** Advanced | **Dataset:** `CartPole-v1` | **Runs on:** CPU, about 2 minute(s)

Update the policy after every step using a learned value function as critic.

## Table of Contents

1. [Learning Objectives](#1.-Learning-Objectives)
2. [Real-World Problem](#2.-Real-World-Problem)
3. [Research Paper](#3.-Research-Paper)
4. [Mathematical Foundation](#4.-Mathematical-Foundation)
5. [Dataset Introduction](#5.-Dataset-Introduction)
6. [Exploratory Data Analysis](#6.-Exploratory-Data-Analysis)
7. [Data Preprocessing](#7.-Data-Preprocessing)
8. [Algorithm Implementation](#8.-Algorithm-Implementation)
9. [Model Training](#9.-Model-Training)
10. [Evaluation](#10.-Evaluation)
11. [Visualization](#11.-Visualization)
12. [Interpretation](#12.-Interpretation)
13. [Experiments and Challenges](#13.-Experiments-and-Challenges)
14. [Research Discussion](#14.-Research-Discussion)
15. [Summary and Further Reading](#15.-Summary-and-Further-Reading)

## 1. Learning Objectives

**After this notebook you will understand:**

- The TD error as an advantage estimate
- Why a critic lowers variance but adds bias
- Two learning rates
- Instability of one step actor critic

**Prerequisites:**

- Probability basics
- Markov decision processes

**Skills you will have afterwards:**

- Run several seeds
- Report learning curves with spread
- Compare with a baseline policy

## 2. Real-World Problem

**Context.** Waiting for the end of an episode wastes information.

**The problem.** Balance CartPole with updates after each step.

**Why it matters.** Actor critic is the structure of most modern policy methods.

**Why machine learning.** Both the policy and the value function are learned.

**Why this algorithm.** One step actor critic is the simplest online version.

## 3. Research Paper

Entries are grouped by type. Bibliographic details marked *verified* were checked against the publisher record (Crossref) or the arXiv abstract page on 2026-11-17. Entries marked *not verified* come from the author's knowledge and must be checked before citing.

### On Actor-Critic Algorithms

- **Type:** foundational paper
- **Authors:** Konda, V. R. and Tsitsiklis, J. N.
- **Year and venue:** 2003, SIAM Journal on Control and Optimization 42(4):1143-1166
- **DOI or URL:** https://doi.org/10.1137/S0363012901385691
- **Verification:** Crossref 2026-11-17
- **Research objective:** Analyse actor critic methods.
- **Methodology:** Two timescale updates of a critic and an actor.
- **Dataset:** Theory.
- **Main findings:** Convergence results for the studied actor critic form.
- **Limitations:** Linear function approximation analysis.

### Reinforcement Learning: An Introduction (2nd ed.)

- **Type:** foundational paper
- **Authors:** Sutton, R. S. and Barto, A. G.
- **Year and venue:** 2018, MIT Press
- **DOI or URL:** http://incompleteideas.net/book/the-book-2nd.html
- **Verification:** not verified in build environment
- **Research objective:** Textbook covering dynamic programming, Monte Carlo, TD, policy gradient.
- **Methodology:** Textbook.
- **Dataset:** Not applicable.
- **Main findings:** Reference text for this section.
- **Limitations:** Textbook, not an experiment.

**Relationship to this notebook.** Konda and Tsitsiklis analyse actor critic convergence under conditions that our small network version does not meet. Sutton and Barto (not verified in the build environment) give the algorithm in textbook form. This notebook does not reproduce the experimental protocol of any listed paper. It uses a different dataset and split, so numbers are not comparable with the paper.

## 4. Mathematical Foundation

**Critic.** Learn $V_w(s)$ with the TD error $\delta_t = r_{t+1} + \gamma V_w(s_{t+1}) - V_w(s_t)$, minimising $\delta_t^2$.

**Actor.** $\theta \leftarrow \theta + \alpha\,\delta_t\,\nabla_\theta \log \pi_\theta(a_t|s_t)$. The TD error is an estimate of the advantage.

**Bias and variance.** The critic bootstraps, so the gradient is biased but has lower variance than REINFORCE.

**Limits.** Two learning rates to tune, instability when the critic is poor.

## 5. Dataset Introduction

**CartPole-v1** (Gymnasium, `classic_control`). A pole is hinged to a cart. State: cart position, cart velocity, pole angle, pole angular velocity (4 real numbers). Actions: push left or right. Reward +1 per step. An episode ends when the pole falls past about 12 degrees, the cart leaves the track, or after 500 steps. The maximum return is 500, and a mean of 475 is the usual solved threshold. There is no dataset, the environment is simulated.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
from IPython.display import display

from ml_lab.rl import tabular, deep
from ml_lab.evaluation import evaluate_policy
from ml_lab.experiments import ExperimentRecord, save_figure
from ml_lab.utils import set_seed, print_environment
from ml_lab import visualization as viz

NOTEBOOK_ID = "actor_critic"
SEED = 42
set_seed(SEED)
viz.apply_style()
print_environment()
print("gymnasium", gym.__version__)

## 6. Exploratory Data Analysis

There is no dataset. We inspect the environment: observation and action spaces, rewards and a random policy baseline. 

In [ ]:
env = gym.make("CartPole-v1")
print("observation space:", env.observation_space)
print("action space:", env.action_space, "| max episode steps:", env.spec.max_episode_steps)
rng0 = np.random.default_rng(0)
random_returns = evaluate_policy(lambda: gym.make("CartPole-v1"), lambda s: int(rng0.integers(2)), episodes=50, seed=0)
print("random policy over 50 episodes:", {k: round(v, 1) for k, v in random_returns.items()})

## 7. Data Preprocessing

No preprocessing: the 4 state values are fed directly to a network with two hidden layers of 64 units. Training uses seeds 0, 1 and 2. Evaluation uses 20 fresh episodes per seed with environment seeds from 1000, never seen in training. Only a CPU is used, so the budgets are small and results will be below long GPU runs.

In [ ]:
TOTAL_STEPS = 60000
SEEDS = [0, 1, 2]
make_env = lambda: gym.make("CartPole-v1")
GRID = np.linspace(1000, TOTAL_STEPS, 40)

## 8. Algorithm Implementation

### 8.1 Concept

The critic says how good the situation is, the actor moves toward actions that turned out better than the critic expected.

### 8.2 Python implementation from scratch

Source of the training function:

In [ ]:
print(inspect.getsource(deep.train_actor_critic))

### 8.3 Library implementation

The algorithm is implemented in `src/ml_lab/rl` without an RL library. Gymnasium supplies only the environment.

### 8.4 Important parameters

| Parameter | Meaning |
|---|---|
| `lr_actor`, `lr_critic` | Learning rates. |
| `gamma` | Discount. |
| `total_steps` | Environment steps. |

## 9. Model Training

Three seeds, 60 000 steps each. One gradient step per environment step for each network.

In [ ]:
runs = {}
for seed in SEEDS:
    runs[seed] = deep.train_actor_critic(seed=seed, total_steps=TOTAL_STEPS)
    last = runs[seed]["returns"][-10:]
    print("seed", seed, "| episodes", len(runs[seed]["returns"]), "| mean training return of last 10 episodes:", round(float(np.mean(last)), 1))

## 10. Evaluation

Greedy policy on 20 fresh episodes per seed. Floor: random policy.

In [ ]:
rng_eval = np.random.default_rng(0)
random_eval = evaluate_policy(make_env, lambda s: int(rng_eval.integers(2)), episodes=20, seed=1000)
evals = {seed: evaluate_policy(make_env, runs[seed]["policy"], episodes=20, seed=1000, success_threshold=475) for seed in SEEDS}
table = pd.DataFrame(evals).T[["mean_return", "std_return", "success_rate"]]
display(table.round(1))
means = np.array([e["mean_return"] for e in evals.values()])
METRICS = {"eval_mean_return": float(means.mean()), "eval_std_across_seeds": float(means.std(ddof=1)), "eval_min_seed": float(means.min()), "eval_max_seed": float(means.max()),
           "random_policy_mean_return": float(random_eval["mean_return"]), "share_of_seeds_solved": float((means >= 475).mean())}
EXTRA = {"per_seed_eval_return": [float(m) for m in means], "training_steps_per_seed": [int(runs[s]["steps"][-1]) for s in SEEDS]}
CONFIG = {"total_steps": TOTAL_STEPS, "lr_actor": 1e-3, "lr_critic": 5e-3, "gamma": 0.99, "seeds": 3}

record = ExperimentRecord(notebook=NOTEBOOK_ID, family="reinforcement_learning", algorithm="ActorCritic", dataset="CartPole-v1",
                          task="control", seed=SEED, config=CONFIG, metrics=METRICS, extra=EXTRA)
print("saved:", record.save())
display(pd.Series(METRICS, name="metric").round(4).to_frame())

## 11. Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
curves = np.array([deep.curve_on_grid(runs[s]["steps"], runs[s]["returns"], GRID, window=10) for s in SEEDS])
for c in curves:
    ax.plot(GRID, c, alpha=0.35)
ax.plot(GRID, np.nanmean(curves, axis=0), color="black", linewidth=2, label="mean of seeds")
ax.axhline(random_eval["mean_return"], color="gray", linestyle=":", label="random policy"); ax.axhline(500, color="green", linestyle="--", label="maximum 500")
ax.set_xlabel("Environment steps"); ax.set_ylabel("Return (mean of last 10 episodes)"); ax.set_title("Learning curves, 3 seeds"); ax.legend()
save_figure(fig, NOTEBOOK_ID, "curves"); plt.show()

## 12. Interpretation

One step actor critic is often unstable, so check the individual seeds.

In [ ]:
print("per seed evaluation return:", np.round(means, 1))
print("a seed counts as solved when its evaluation mean is at least 475; solved seeds:", int((means >= 475).sum()), "of", len(SEEDS))
final_train = np.array([np.mean(runs[s]["returns"][-10:]) for s in SEEDS])
print("training return of last 10 episodes versus evaluation return:", np.round(final_train, 1), "vs", np.round(means, 1))

## 13. Experiments and Challenges

### Experiment 1: Critic learning rate

Slow, equal and fast critic. Two seeds and 25 000 steps.

In [ ]:
for name, kw in [("1e-3", dict(lr_critic=1e-3)), ("5e-3", dict(lr_critic=5e-3)), ("2e-2", dict(lr_critic=2e-2))]:
    rets = []
    for seed in [0, 1]:
        out = deep.train_actor_critic(seed=seed, total_steps=25000, **kw)
        ev = evaluate_policy(make_env, out["policy"], episodes=10, seed=1000)
        rets.append(ev["mean_return"])
    print(f"{name}: evaluation return by seed {np.round(rets, 1)}, mean {np.mean(rets):.1f}")

### Experiment 2: Actor learning rate

3e-4, 1e-3 and 3e-3. Two seeds and 25 000 steps.

In [ ]:
for name, kw in [("3e-4", dict(lr_actor=3e-4)), ("1e-3", dict(lr_actor=1e-3)), ("3e-3", dict(lr_actor=3e-3))]:
    rets = []
    for seed in [0, 1]:
        out = deep.train_actor_critic(seed=seed, total_steps=25000, **kw)
        ev = evaluate_policy(make_env, out["policy"], episodes=10, seed=1000)
        rets.append(ev["mean_return"])
    print(f"{name}: evaluation return by seed {np.round(rets, 1)}, mean {np.mean(rets):.1f}")

### Experiment 3: Discount factor

0.9 versus 0.99. Two seeds and 25 000 steps.

In [ ]:
for name, kw in [("0.9", dict(gamma=0.9)), ("0.99", dict(gamma=0.99))]:
    rets = []
    for seed in [0, 1]:
        out = deep.train_actor_critic(seed=seed, total_steps=25000, **kw)
        ev = evaluate_policy(make_env, out["policy"], episodes=10, seed=1000)
        rets.append(ev["mean_return"])
    print(f"{name}: evaluation return by seed {np.round(rets, 1)}, mean {np.mean(rets):.1f}")

### Exercises for learners

1. Repeat with a different set of seeds and report the spread.
2. Change the discount factor and describe what happens to the learned behaviour.
3. Change one hyperparameter by a factor of 10 and compare learning curves.

## 14. Research Discussion

Konda and Tsitsiklis show convergence under specific conditions (two time scales). Our learning rate experiments look at the ratio of the two rates in practice.

**Dataset and method differences.** The papers above use other data and protocols. We do not claim that our numbers agree or disagree with them. This notebook only shows whether the behaviour they describe appears on our data.

**Possible improvements.** Use n step returns or GAE, batch several environments (A2C), entropy bonus and more seeds.

**Research limitations.** One dataset and one seed for the headline numbers. Three seeds and a CPU sized step budget, so these runs show trends, not converged scores.

## 15. Summary and Further Reading

**Key takeaways**

- The critic gives a low variance advantage.
- Two learning rates must be balanced.
- Check seeds one by one.

**Use cases.** Online control.

**Limitations.** Unstable, biased.

**Recommended next notebook.** PPO.

**Further reading.** See `references.md` in this folder.